# MiniMax H3 Ref2VA Portrait Video Launcher (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/j7708git/colab-workbench/blob/main/comfyui/minimax_h3.ipynb)

本 Notebook 專為 **MiniMax H3 參考人像生成影片＋原生雙聲道音訊（Ref2VA）** 設計，整合 `ComfyUI`（≥ v0.38.0）、`ComfyUI-KJNodes` 無損分塊節點與預載工作流 `minimax_h3_ref2va_8step_kj.json`：

- **硬體與執行階段建議**：強烈建議選用 **A100（40GB/80GB，High-RAM）** 或 **L4（24GB，High-RAM）**。不建議使用 T4（16GB VRAM 無高階加速且標準 RAM 不易承載 ~41.4 GB 權重與系統記憶體緩衝）。
- **嚴禁交叉混用模式與 LoRA**：參考人像生影片務必使用 `minimax_h3_ref2va_*` 主模型搭配 `minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors`（不可混用 `fl2v` LoRA）；若只需首尾幀圖生影請改選 `minimax_h3_fl2va_8step` preset 並配 `MiniMaxH3ImageToVideo` 工作流。
- **零磁碟 `/dev/shm` 輸出與延長保留**：輸出影片與日誌皆駐留於記憶體磁碟 `/dev/shm/comfy-out` 與 `/dev/shm/comfy-tmp`，並預設將過期清理延長為 `1800` 秒（30 分鐘），確保瀏覽器有足夠時間預覽與下載 MP4。
- **Colab Secrets**：建議於左側鑰匙圖示設定 `HF_TOKEN` 並開啟 **Notebook access**，以免 Colab 共享 IP 下載約 41.4 GB 權重時觸發 HTTP 429 限流。

In [ ]:
#@title 1. 同步 colab-workbench、載入 HF_TOKEN 並檢查 GPU／RAM／磁碟空間
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/j7708git/colab-workbench.git"
REPO_DIR = "/content/colab-workbench"
FALLBACK_BRANCH = "main"
BRANCH = "main"  #@param {type:"string"}
BRANCH = (BRANCH or FALLBACK_BRANCH).strip() or FALLBACK_BRANCH

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(
    ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
    check=True,
)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

init_path = os.path.join(REPO_DIR, "comfyui", "__init__.py")
wf_path = os.path.join(REPO_DIR, "comfyui", "workflows", "minimax_h3_ref2va_8step_kj.json")
if (not os.path.isfile(init_path) or not os.path.isfile(wf_path)) and BRANCH != FALLBACK_BRANCH:
    print(f"[minimax-h3] 分支 '{BRANCH}' 尚未包含 MiniMax H3 檔案，自動切換至 {FALLBACK_BRANCH} ...")
    BRANCH = FALLBACK_BRANCH
    subprocess.run(
        ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
        check=True,
    )
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

assert os.path.isfile(init_path), f"找不到 {init_path}，請確認分支 {BRANCH} 內容是否完整"
assert os.path.isfile(wf_path), f"找不到 {wf_path}，請確認分支 {BRANCH} 已包含 MiniMax H3 工作流"

sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
for _mod in [
    k
    for k in list(sys.modules)
    if k in {"comfyui", "shared"} or k.startswith(("comfyui.", "shared."))
]:
    del sys.modules[_mod]
importlib.invalidate_caches()

import comfyui
from shared import export_secret_to_env, get_secret_status

assert getattr(comfyui, "__file__", None) is not None and hasattr(comfyui, "download_preset_models"), (
    f"comfyui 載入異常（__file__={getattr(comfyui, '__file__', None)}）"
)
head_sha = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
has_hf_secret, hf_secret_status = get_secret_status("HF_TOKEN")
hf_token = export_secret_to_env("HF_TOKEN", required=False)
print("BRANCH:", BRANCH, f"({head_sha})")
print("comfyui.__file__:", comfyui.__file__)
if hf_token:
    print(f"HF_TOKEN: 已載入（來源：{hf_secret_status}）")
elif "disabled" in hf_secret_status or "access" in hf_secret_status:
    print("HF_TOKEN: 偵測到 Secret 但「Notebook access」未開啟！請至 Colab 左側鑰匙圖示 Secrets 開啟 Notebook access 開關，避免匿名下載觸發 429 限流。")
else:
    print(f"HF_TOKEN: 未設定（{hf_secret_status}）；公開模型可匿名下載，但建議設定 HF_TOKEN 以避免 429 限流。")

# Preflight: GPU / VRAM / System RAM / Disk / PyTorch
try:
    import torch
    print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        vram_gb = props.total_memory / (1024 ** 3)
        print(f"GPU: {props.name} ({vram_gb:.1f} GiB VRAM)")
        if "T4" in props.name.upper():
            print("警告：偵測到 T4 GPU，建議改用 L4 或 A100（High-RAM）以穩定運行 MiniMax H3 Ref2VA。")
    else:
        print("警告：目前未偵測到 CUDA GPU，請於 Colab 選單『執行階段 -> 變更執行階段類型』選擇 L4 或 A100。")
except Exception as exc:
    print(f"PyTorch 檢查失敗：{exc}")

try:
    with open("/proc/meminfo", encoding="utf-8") as fp:
        for line in fp:
            if line.startswith("MemTotal:"):
                ram_gb = int(line.split()[1]) / (1024 ** 2)
                print(f"System RAM: {ram_gb:.1f} GiB")
                if ram_gb < 24:
                    print("警告：系統記憶體低於 24 GiB，建議改選 Colab High-RAM 執行階段。")
                break
except OSError:
    pass

for label, path in (("/content", "/content"), ("/dev/shm", "/dev/shm")):
    if os.path.exists(path):
        usage = shutil.disk_usage(path)
        free_gb = usage.free / (1024 ** 3)
        total_gb = usage.total / (1024 ** 3)
        print(f"Disk {label}: {free_gb:.1f} / {total_gb:.1f} GiB free")
        if label == "/content" and free_gb < 55:
            print("警告：/content 可用磁碟低於 55 GiB，下載約 41.4 GB 權重＋ComfyUI 可能空間吃緊。")

In [ ]:
#@title 2. 安裝／更新 ComfyUI、ComfyUI-KJNodes（無損分塊）與 ffmpeg
import os
import subprocess

INSTALL_KJNODES = True  #@param {type:"boolean"}
UPGRADE_TORCH_CU130 = False  #@param {type:"boolean"}

env = os.environ.copy()
env["INSTALL_KJNODES"] = "1" if INSTALL_KJNODES else "0"
env["UPGRADE_TORCH_CU130"] = "1" if UPGRADE_TORCH_CU130 else "0"
env["INSTALL_WORKFLOWS"] = "1"

print(f"開始執行 install.sh（INSTALL_KJNODES={env['INSTALL_KJNODES']}, UPGRADE_TORCH_CU130={env['UPGRADE_TORCH_CU130']}）...")
subprocess.run(["bash", "/content/colab-workbench/comfyui/install.sh"], env=env, check=True)

In [ ]:
#@title 3. 下載 MiniMax H3 模型套件並安裝內建 Ref2VA 工作流
import importlib
import os
import shutil
import sys

REPO_DIR = "/content/colab-workbench"
sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
for _pkg, _attr in (("comfyui", "download_preset_models"), ("shared", "get_secret_status")):
    _cached = sys.modules.get(_pkg)
    if _cached is not None and (
        getattr(_cached, "__file__", None) is None
        or not hasattr(_cached, _attr)
    ):
        for _mod in [k for k in list(sys.modules) if k == _pkg or k.startswith(f"{_pkg}.")]:
            del sys.modules[_mod]
importlib.invalidate_caches()

PRESET = "minimax_h3_ref2va_8step"  #@param ["minimax_h3_ref2va_8step", "minimax_h3_ref2va_full", "minimax_h3_ref2va_8step_fp8", "minimax_h3_fl2va_8step"]
EXTRA_MODELS_YAML = ""  #@param {type:"string"}

from comfyui import download_preset_models, install_bundled_workflows, load_models_from_yaml
from shared import get_secret_status

has_hf_secret, hf_secret_status = get_secret_status("HF_TOKEN")
if has_hf_secret:
    print(f"HF_TOKEN 狀態：已啟用（{hf_secret_status}）")
else:
    print(f"HF_TOKEN 狀態：匿名模式（{hf_secret_status}；若遇 429 限流請確認 Colab Secrets 的 Notebook access 已開啟）")

extra = load_models_from_yaml(EXTRA_MODELS_YAML) if EXTRA_MODELS_YAML.strip() else None
downloaded = download_preset_models(preset=PRESET, extra_entries=extra)
installed_wfs = install_bundled_workflows()

print(f"模型就緒（preset={PRESET}，共 {len(downloaded)} 個檔案）:")
for p in downloaded:
    print(" -", p)
print(f"工作流已同步至 ComfyUI（共 {len(installed_wfs)} 份）:")
for wf in installed_wfs:
    print(" -", wf)
if os.path.exists("/content"):
    free_gb = shutil.disk_usage("/content").free / (1024 ** 3)
    print(f"/content 剩餘磁碟空間：{free_gb:.1f} GiB")

In [ ]:
#@title 4. 啟動零磁碟 ComfyUI（--reserve-vram 2.0）與 Cloudflare Tunnel
import importlib
import shlex
import sys

REPO_DIR = "/content/colab-workbench"
sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
for _pkg, _attr in (("comfyui", "launch_comfyui_zero_disk"), ("shared", "get_secret_status")):
    _cached = sys.modules.get(_pkg)
    if _cached is not None and (
        getattr(_cached, "__file__", None) is None
        or not hasattr(_cached, _attr)
    ):
        for _mod in [k for k in list(sys.modules) if k == _pkg or k.startswith(f"{_pkg}.")]:
            del sys.modules[_mod]
importlib.invalidate_caches()

from comfyui import launch_comfyui_zero_disk

PORT = 8188  #@param {type:"integer"}
RESERVE_VRAM_GB = 2.0  #@param {type:"number"}
MAX_AGE_SEC = 1800  #@param {type:"integer"}
REAPER_INTERVAL_SEC = 120  #@param {type:"integer"}
TUNNEL_PROTOCOL = "http2"  #@param ["http2", "auto", "quic"]
EXTRA_ARGS = ""  #@param {type:"string"}

session = launch_comfyui_zero_disk(
    port=PORT,
    max_age_sec=MAX_AGE_SEC,
    reaper_interval_sec=REAPER_INTERVAL_SEC,
    tunnel_protocol=TUNNEL_PROTOCOL,
    reserve_vram_gb=RESERVE_VRAM_GB if RESERVE_VRAM_GB > 0 else None,
    extra_args=shlex.split(EXTRA_ARGS) if EXTRA_ARGS.strip() else None,
)

print("=" * 68)
print("MiniMax H3 Ref2VA ComfyUI 服務已上線！")
print("Public URL:       ", session.public_url)
print("Reserve VRAM (GB):", RESERVE_VRAM_GB if RESERVE_VRAM_GB > 0 else "disabled")
print("RAM Disk 保留時間:", f"{MAX_AGE_SEC} 秒（{MAX_AGE_SEC / 60:.0f} 分鐘）")
print("Output RAM Disk:  ", session.output_dir)
print("ComfyUI Log:      ", session.log_file)
print("Tunnel Log:       ", session.tunnel_log_file)
print("-" * 68)
print("使用步驟：")
print("1. 點擊上方 Public URL 開啟 ComfyUI 網頁。")
print("2. 於左側欄點選 Workflows，開啟『minimax_h3_ref2va_8step_kj.json』。")
print("3. 在『Reference Portrait 1 (<Picture 1>)』上傳人像照並編輯提示詞，按 Queue 開始生成。")
print("4. 影片生成完成後請即時於 UI 下載 MP4（預設在 /dev/shm 保留 30 分鐘）。")
print("5. 結束使用請執行：session.stop()")
print("=" * 68)

In [ ]:
#@title 5. 輔助工具：查看日誌、輸出暫存與停止服務（含提示詞範本）
from pathlib import Path

# 需要關閉服務與隧道時，取消下一行註解並執行：
# session.stop()

if "session" in globals():
    log_path = Path(session.log_file)
    if log_path.is_file():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("=== ComfyUI Log (last 30 lines) ===")
        for line in lines[-30:]:
            print(line)
    out_dir = Path(session.output_dir)
    if out_dir.is_dir():
        files = sorted(p for p in out_dir.rglob("*") if p.is_file())
        print(f"\n=== /dev/shm/comfy-out ({len(files)} file(s)) ===")
        for f in files:
            print(f" - {f.relative_to(out_dir)} ({f.stat().st_size / (1024 ** 2):.2f} MiB)")
else:
    print("尚未啟動 session；請先執行第 4 格。")

PROMPT_TEMPLATE = """integrated_multimodal_description: [Shot 1] Medium waist-up portrait of the person in <Picture 1>, preserving their exact facial features, hairstyle, skin tone, and clothing. In warm afternoon window light inside a cozy minimalist studio, the subject smiles naturally and says gently: <d>[Chinese] 你好，今天一起看看這段畫面吧。</d> Keep the subject's head occupying a steady medium proportion from first frame to last frame. Smooth cinematic handheld motion.

overall_soundscape: Clear natural voice, soft room ambience, subtle clothing rustle.

non_diegetic_music: Gentle acoustic piano and warm ambient pad."""

print("\n=== 參考人像三段式英文提示詞範本 ===")
print(PROMPT_TEMPLATE)